In [1]:
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
import numpy as np

## Q1. Downloading the data

We'll use [the same NYC taxi dataset](https://www1.nyc.gov/site/tlc/about/tlc-trip-record-data.page),
but instead of "**Green** Taxi Trip Records", we'll use "**Yellow** Taxi Trip Records".

Download the data for January and February 2023.

Read the data for January. How many columns are there?

## Datasets

In [4]:
YELLOW_TAXI_TRIP_RECORDS_URL_JAN = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
YELLOW_TAXI_TRIP_RECORDS_URL_FEB = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-02.parquet"

In [4]:
jan_df = pd.read_parquet(YELLOW_TAXI_TRIP_RECORDS_URL_JAN)
jan_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1.0,0.97,1.0,N,161,141,2,9.3,1.00,0.5,0.00,0.0,1.0,14.30,2.5,0.00
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1.0,1.10,1.0,N,43,237,1,7.9,1.00,0.5,4.00,0.0,1.0,16.90,2.5,0.00
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1.0,2.51,1.0,N,48,238,1,14.9,1.00,0.5,15.00,0.0,1.0,34.90,2.5,0.00
3,1,2023-01-01 00:03:48,2023-01-01 00:13:25,0.0,1.90,1.0,N,138,7,1,12.1,7.25,0.5,0.00,0.0,1.0,20.85,0.0,1.25
4,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1.0,1.43,1.0,N,107,79,1,11.4,1.00,0.5,3.28,0.0,1.0,19.68,2.5,0.00


In [5]:
jan_df.shape[1]

19

There are **19** columns

## Q2. Computing duration

Now let's compute the `duration` variable. It should contain the duration of a ride in minutes. 

What's the standard deviation of the trips duration in January?

In [6]:
jan_df["tpep_pickup_datetime"] = pd.to_datetime(jan_df["tpep_pickup_datetime"])
jan_df["tpep_dropoff_datetime"] = pd.to_datetime(jan_df["tpep_dropoff_datetime"])
jan_df.loc[:, "duration"] = (jan_df["tpep_dropoff_datetime"] - jan_df["tpep_pickup_datetime"]).dt.total_seconds() / 60.0
float(jan_df["duration"].std().round(2))

42.59

The standard deviation of the trips duration in January is $42.59$

## Q3. Dropping outliers

Next, we need to check the distribution of the `duration` variable. There are some outliers. Let's remove them and keep only the records where the duration was between 1 and 60 minutes (inclusive).

What fraction of the records left after you dropped the outliers?

In [7]:
jan_good_duration_df = jan_df[(jan_df["duration"] >=1) & (jan_df["duration"] <= 60)]
jan_good_duration_df.shape[0] / jan_df.shape[0] * 100

98.1220282212598

The fraction of the records left after dropping outliers is 98.12%

## Q4. One-hot encoding

Let's apply one-hot encoding to the pickup and dropoff location IDs. We'll use only these two features for our model. 

* Turn the dataframe into a list of dictionaries (remember to re-cast the ids to strings - otherwise it will 
  label encode them)
* Fit a dictionary vectorizer 
* Get a feature matrix from it

What's the dimensionality of this matrix (number of columns)?

In [8]:
categorical_features = [
    "PULocationID",
    "DOLocationID"
]
jan_good_duration_df.loc[:, categorical_features] = jan_good_duration_df[categorical_features].astype(str)
train_dict = jan_good_duration_df[categorical_features].to_dict(orient="records")

dv = DictVectorizer()
X_train = dv.fit_transform(train_dict)
X_train

/tmp/ipykernel_10949/1132647559.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['161' '43' '48' ... '114' '230' '262']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  jan_good_duration_df.loc[:, categorical_features] = jan_good_duration_df[categorical_features].astype(str)
/tmp/ipykernel_10949/1132647559.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['141' '237' '238' ... '239' '79' '143']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  jan_good_duration_df.loc[:, categorical_features] = jan_good_duration_df[categorical_features].astype(str)


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 6018346 stored elements and shape (3009173, 515)>

The number of features is $515$.

## Q5. Training a model

Now let's use the feature matrix from the previous step to train a model. 

* Train a plain linear regression model with default parameters, where duration is the response variable
* Calculate the RMSE of the model on the training data

What's the RMSE on train?

In [10]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

In [10]:
y_train = jan_good_duration_df["duration"].values

lr = LinearRegression()
lr.fit(X_train, y_train)

y_train_pred = lr.predict(X_train)

rmse = root_mean_squared_error(y_true=y_train, y_pred=y_train_pred)
rmse

7.6492624397080675

The RMSE on the training dataset is $7.65$

## Q6. Evaluating the model

Now let's apply this model to the validation dataset (February 2023). 

What's the RMSE on validation?

In [2]:
def read_and_process_dataset(file: str) -> pd.DataFrame:
    df = pd.read_parquet(file)
    df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
    df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])
    df.loc[:, "duration"] = (df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]).dt.total_seconds() / 60.0
    df = df[(df["duration"] >=1) & (df["duration"] <= 60)]
    categorical_features = [
        "PULocationID",
        "DOLocationID"
    ]
    df.loc[:, categorical_features] = df[categorical_features].astype(str)
    return df



In [5]:
df_train = read_and_process_dataset(file=YELLOW_TAXI_TRIP_RECORDS_URL_JAN)

/tmp/ipykernel_12332/3574364696.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['161' '43' '48' ... '114' '230' '262']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, categorical_features] = df[categorical_features].astype(str)
/tmp/ipykernel_12332/3574364696.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['141' '237' '238' ... '239' '79' '143']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:, categorical_features] = df[categorical_features].astype(str)


In [6]:
df_val = read_and_process_dataset(file=YELLOW_TAXI_TRIP_RECORDS_URL_FEB)

/tmp/ipykernel_12332/3574364696.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['142' '132' '161' ... '158' '79' '161']' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:, categorical_features] = df[categorical_features].astype(str)
/tmp/ipykernel_12332/3574364696.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['163' '26' '145' ... '143' '162' '140']' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:, categorical_features] = df[categorical_features].astype(str)


In [8]:
categorical_features = [
    "PULocationID",
    "DOLocationID"
]

train_dict = df_train[categorical_features].to_dict(orient="records")
val_dict = df_val[categorical_features].to_dict(orient="records")

dv = DictVectorizer()

X_train = dv.fit_transform(train_dict)
X_val = dv.transform(val_dict)

y_train = df_train["duration"].values
y_val = df_val["duration"].values

In [11]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_train_pred = lr.predict(X_train)
y_val_pred = lr.predict(X_val)

In [12]:
train_rmse = root_mean_squared_error(y_true=y_train, y_pred=y_train_pred)
val_rmse = root_mean_squared_error(y_true=y_val, y_pred=y_val_pred)
val_rmse

7.81181211389241

The RMSE on the validation dataset is $7.81$